# augmented-v4: gündelik dil verisiyle son eğitim turu

**Çalışma zamanı → T4 GPU → Tümünü çalıştır.** Üç seed, her biri yaklaşık 1 saat. Her seed bitince Drive'a kaydedilir;
bağlantı koparsa ya da GPU kotası biterse yeniden **Tümünü çalıştır**: hazır olanlar atlanır.
Mühürlü test burada çalışmaz; yeni mühürlü set (sealed2) etiketlenince ayrı defterle bir kez ölçülür.


In [ ]:
import os, pathlib, shutil, subprocess, zipfile, json, torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı türünü T4 GPU yap."
from google.colab import drive
drive.mount('/content/drive')
# Klasörler başka bir hesaptan paylaşılıp kısayol olarak eklenmiş olabilir: kısayol hedeflerinde de ara
ROOTS = [pathlib.Path('/content/drive/MyDrive'), pathlib.Path('/content/drive/.shortcut-targets-by-id')]
def walk_dirs():
    for r in ROOTS:
        if r.exists():
            for d, subdirs, files in os.walk(r, followlinks=True):
                yield pathlib.Path(d), subdirs, files
DRIVE = pathlib.Path('/content/drive/MyDrive/turkce-llm-dlp')
if not DRIVE.exists():
    DRIVE = next((d / 'turkce-llm-dlp' for d, sub, _ in walk_dirs() if 'turkce-llm-dlp' in sub), DRIVE)
assert (DRIVE / 'runs').exists(), f'{DRIVE}/runs bulunamadı: turkce-llm-dlp klasörünü bu hesapta Drive\'ım\'a kısayol olarak ekle'
print('Drive klasörü:', DRIVE)
os.chdir('/content')
shutil.rmtree('/content/proje', ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/irmakyaprakk/turkce-llm-dlp.git', '/content/proje'], check=True)
bundles = set()
for d, _, files in walk_dirs():
    if 'BUNDLE_INFO.txt' in files: bundles.add(d)
    bundles |= {d / f for f in files if f.startswith('turkce-dlp-colab-bundle') and f.endswith('.zip')}
for src in sorted(bundles, key=lambda p: p.name):
    tmp = pathlib.Path('/content/_paket'); shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src, tmp) if src.is_dir() else zipfile.ZipFile(src).extractall(tmp)
    root = next((p.parent for p in tmp.rglob('BUNDLE_INFO.txt')), tmp)
    if (root / 'data').exists():
        shutil.copytree(root / 'data', '/content/proje/data', dirs_exist_ok=True)
os.chdir('/content/proje')
!pip -q install -r training/ner/requirements.txt
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# augmented-v4 verisi: Drive'daki v6 paketinden gelir (yoksa kaynaklardan yeniden üretilir); dondurulan sürümle doğrulanır
import hashlib
if not pathlib.Path('data/ner/processed/augmented-v4/train.jsonl').exists():
    for need in ('data/ner/processed/person-org-location-v1/train.jsonl', 'data/ner/processed/openstreetmap-tr/splits/train.jsonl',
                 'data/ner/processed/real-address-v2/test.jsonl'):
        assert pathlib.Path(need).exists(), f"augmented-v4 yok: 'turkce-dlp-colab-bundle-v6' paketini Drive'a yükle ({need} de bulunamadı)"
    !python training/ner/prepare_augmented.py --osm-original-ratio 0.15 --v3-contexts 10000 800 --mention-replace 5000 300 --v4-informal 12000 800 --output-dir data/ner/processed/augmented-v4 > /dev/null
EXPECTED = {'train': 'e91b955188a066c97bd78cfec46733148a1201fb74d315c5165285c86f0750ec', 'dev': 'a51a72a42ca1492b8cb07856dccc786a548ee3308f515e5932f3f81d6276b2e5'}
for split, digest in EXPECTED.items():
    got = hashlib.sha256(pathlib.Path(f'data/ner/processed/augmented-v4/{split}.jsonl').read_bytes()).hexdigest()
    assert got == digest, f'{split} verisi dondurulan sürümle aynı değil ({got[:12]})'
print('augmented-v4 doğrulandı:', json.load(open('data/ner/processed/augmented-v4/quality_report.json'))['rows'])


In [ ]:
# Üç seed eğitimi (her biri ~1 saat). Hazır olan atlanır; kesilirse yeniden "Tümünü çalıştır".
for seed in (42, 43, 44):
    saved = DRIVE / 'runs' / f'augmented_v4_seed{seed}'
    if (saved / 'model').exists():
        print(f'seed {seed}: hazır, atlanıyor'); continue
    out = f'/content/runs/augmented_v4_seed{seed}'
    print(f'seed {seed}: eğitim başlıyor')
    !USE_TF=0 python training/ner/train.py --data-dir data/ner/processed/augmented-v4 --labels-path training/ner/labels.json --output-dir {out} --epochs 3 --seed {seed} --fp16 > /content/train-{seed}.log 2>&1
    !tail -3 /content/train-{seed}.log
    shutil.rmtree(saved, ignore_errors=True)
    shutil.copytree(out, saved, ignore=shutil.ignore_patterns('checkpoints'))
    print(f'seed {seed}: Drive\'a kaydedildi')


In [ ]:
# Geliştirme seti (realistic-v2) — yalnız bilgi amaçlı; karar yeni mühürlü sette (sealed2) verilecek.
!python training/ner/build_gold_set.py --out-dir data/ner/processed/realistic-v2 > /dev/null
files = []
for seed in (42, 43, 44):
    model = DRIVE / 'runs' / f'augmented_v4_seed{seed}' / 'model'
    preds = f'/content/gelistirme-v4-{seed}.jsonl'
    open(preds, 'w').close()
    for split in ('dev', 'test'):
        !USE_TF=0 python training/ner/evaluate_model.py --model-dir "{model}" --data-dir data/ner/processed/realistic-v2 --split {split} --dump-predictions /content/p.jsonl > /dev/null
        !cat /content/p.jsonl >> {preds}
    target = DRIVE / 'runs' / f'augmented_v4_seed{seed}' / 'gelistirme-pipeline.json'
    !python training/ner/evaluate_rules.py --data-dir data/ner/processed/realistic-v2 --splits dev test --with-predictions {preds} --output-json "{target}" > /dev/null
    files.append(str(target))
out = subprocess.run(['python', 'training/ner/aggregate_seeds.py', *files], capture_output=True, text=True, check=True).stdout
(DRIVE / 'runs' / 'gelistirme-v4-summary.json').write_text(out)
m = json.loads(out)['metrics']
print(json.dumps({k.split('.')[-1]: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in m.items()
                  if k.startswith('pipeline_model_plus_rules.') and any(x in k for x in ('_f1', 'overall_precision', '_coverage'))},
                 ensure_ascii=False, indent=1))
